# 06. Радиус отсечения: энергия, сила и непрерывность

**Слайд 12. Время:** 25 минут. Сравним буквальное отсечение, сдвиг потенциала и плавное переключение. Для прозрачности используются `CustomBondForce` и две частицы; это демонстрация математики cutoff, а не ускорения списка соседей. В NVE-части фиксированный центр и частица в слабом гармоническом удерживающем потенциале обеспечивают повторные пересечения границы.

**Запуск:** Colab → «Среда выполнения → Выполнить все». Выбирать GPU не нужно. Все исходные системы создаются кодом; загрузка PDB и ключи доступа не требуются. Выполняйте ячейки сверху вниз. После изменения параметров повторите зависимые ячейки.

**Обозначения:** энергии в kJ/mol, длины в nm, время в ps; 1 fs = 0.001 ps.

In [ ]:
# Установка фиксированной версии: CPU достаточно, GPU не требуется.
%pip -q install openmm==8.3.1 numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmm as mm
from openmm import unit
from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation
import platform
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True, 'font.size': 11})
rng = np.random.default_rng(2026)
# Маленькие учебные системы: Reference исключает накладные расходы GPU.
backend = mm.Platform.getPlatformByName('Reference')
R = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole/unit.kelvin)
print('OpenMM', mm.__version__, '| Python', platform.python_version(), '|', backend.getName())
# Числовые единицы ниже: nm, ps, dalton, kJ/mol, K.
def vv(dt):
    it = mm.CustomIntegrator(dt*unit.picoseconds)
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    it.addComputePerDof('x', 'x+dt*v')
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    return it

def state(ctx):
    st = ctx.getState(getPositions=True, getVelocities=True, getEnergy=True)
    x = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    v = st.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)
    u = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    return x, v, u

def movie(frames, title='', bonds=(), interval=60):
    """Небольшая XY-проекция. Показанные траектории могут быть трёхмерными."""
    a = np.asarray(frames)[::max(1, len(frames)//100)]
    fig, ax = plt.subplots(figsize=(5, 4))
    lo=a[:,:,:2].min(axis=(0,1)); hi=a[:,:,:2].max(axis=(0,1))
    pad=max(float(np.max(hi-lo))*0.15,0.1)
    ax.set(xlim=(lo[0]-pad,hi[0]+pad),ylim=(lo[1]-pad,hi[1]+pad),
           xlabel='x, nm',ylabel='y, nm',title=title,aspect='equal')
    dots=ax.scatter(a[0,:,0],a[0,:,1],s=45,c=np.arange(a.shape[1]),cmap='viridis')
    lines=[ax.plot([],[],color='gray')[0] for _ in bonds]
    def update(i):
        dots.set_offsets(a[i,:,:2])
        for line,(j,k) in zip(lines,bonds): line.set_data(a[i,[j,k],0],a[i,[j,k],1])
        return [dots]+lines
    ani=FuncAnimation(fig,update,frames=len(a),interval=interval,blit=True)
    html=ani.to_jshtml(); plt.close(fig); display(HTML(html))


## Три определения потенциала

Переключающая функция $S=1-10t^3+15t^4-6t^5$ на интервале от ron до rc; вне интервала она равна 1 или 0. OpenMM автоматически дифференцирует весь потенциал, включая S.

In [ ]:
sig,eps,rc,ron=.34,1.,.70,.50
uc=4*eps*((sig/rc)**12-(sig/rc)**6)
base='4*eps*((sig/r)^12-(sig/r)^6)'
exprs={'cut':f'step(rc-r)*({base})',
       'shift':f'step(rc-r)*(({base})-uc)',
       'switch':f'({base})*(1-10*t^3+15*t^4-6*t^5);t=min(1,max(0,(r-ron)/(rc-ron)))'}
def cutoff_context(kind,dt=.001,trap=False):
    sys=mm.System(); sys.addParticle(0); sys.addParticle(40)
    f=mm.CustomBondForce(exprs[kind])
    for key,val in [('sig',sig),('eps',eps),('rc',rc),('ron',ron),('uc',uc)]:f.addGlobalParameter(key,val)
    f.addBond(0,1,[]); sys.addForce(f)
    if trap:
        g=mm.CustomExternalForce('0.5*20*(x-0.65)^2');g.addParticle(1,[]);sys.addForce(g)
    # vv делит на массу: для фиксированной частицы обновления маскируются.
    it=mm.CustomIntegrator(dt)
    it.addComputePerDof('v','v+0.5*dt*f/(m+delta(m))')
    it.addComputePerDof('x','x+dt*v*step(m-0.1)')
    it.addComputePerDof('v','(v+0.5*dt*f/(m+delta(m)))*step(m-0.1)')
    ctx=mm.Context(sys,it,backend);return ctx,it
rs=np.linspace(.45,.85,600);fig,ax=plt.subplots(1,2,figsize=(12,4))
for kind in exprs:
    ctx,it=cutoff_context(kind); us=[]; fs=[]
    for r in rs:
        ctx.setPositions([[0,0,0],[r,0,0]])
        s=ctx.getState(getEnergy=True,getForces=True)
        us.append(s.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole))
        fs.append(s.getForces(asNumpy=True).value_in_unit(unit.kilojoule_per_mole/unit.nanometer)[1,0])
    ax[0].plot(rs,us,label=kind);ax[1].plot(rs,fs,label=kind)
    del ctx,it
for a in ax:a.axvline(rc,color='gray',ls=':');a.set_xlabel('r, nm');a.legend()
ax[0].set_ylabel('U, kJ/mol');ax[1].set_ylabel('F, kJ/mol/nm');plt.show()


## Пересечения границы в динамике

У резкого отсечения формальная производная включает сингулярный вклад на границе. Обычная MD-сила через step его не воспроизводит: поэтому скачок энергии здесь ожидаем. При сдвиге исчезает скачок U, но остаётся скачок силы. Плавное переключение меняет силы во всём интервале переключения.

In [ ]:
# Выберите 'cut', 'shift' или 'switch' и повторите эту ячейку.
visual_mode = 'switch'
visual_dt = 0.001          # ps
visual_duration = 12.0     # ps
visual_stride = 20        # сохраняем каждые 0.02 ps

assert visual_mode in exprs
ctx_vis, it_vis = cutoff_context(visual_mode, dt=visual_dt, trap=True)
# Полную потенциальную энергию получаем непосредственно из OpenMM.
x_grid = np.linspace(0.30, 1.05, 450)
U_total = []
for x_pos in x_grid:
    ctx_vis.setPositions([[0, 0, 0], [x_pos, 0, 0]])
    U_total.append(ctx_vis.getState(getEnergy=True).getPotentialEnergy()
                   .value_in_unit(unit.kilojoule_per_mole))
U_total = np.asarray(U_total)
U_trap = 0.5 * 20.0 * (x_grid - 0.65)**2
U_pair = U_total - U_trap

ctx_vis.setPositions([[0, 0, 0], [0.55, 0, 0]])
ctx_vis.setVelocities([[0, 0, 0], [0.22, 0, 0]])
records = []
for j in range(round(visual_duration / (visual_dt * visual_stride)) + 1):
    st = ctx_vis.getState(getPositions=True, getVelocities=True,
                         getEnergy=True, getForces=True)
    x_pos = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)[1, 0]
    velocity = st.getVelocities(asNumpy=True).value_in_unit(
        unit.nanometer / unit.picosecond)[1, 0]
    potential = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    force = st.getForces(asNumpy=True).value_in_unit(
        unit.kilojoule_per_mole / unit.nanometer)[1, 0]
    kinetic = 0.5 * 40.0 * velocity**2
    records.append([j*visual_dt*visual_stride, x_pos, velocity,
                    potential, kinetic, force])
    it_vis.step(visual_stride)
del ctx_vis, it_vis
records = np.asarray(records)
t_vis, x_vis, v_vis, u_vis, k_vis, f_vis = records.T
e_vis = u_vis + k_vis
assert np.all(np.isfinite(records))
assert np.any(x_vis < rc) and np.any(x_vis > rc), 'Траектория не пересекла cutoff.'

fig_vis, (ax_U, ax_x, ax_E) = plt.subplots(
    3, 1, figsize=(10, 9), gridspec_kw={'height_ratios': [3, 1.4, 2]}
)
fig_vis.suptitle(f'Движение в полном потенциале: {visual_mode}', fontsize=14)

# Верхняя панель: энергетический ландшафт, не реальная высота частицы.
ax_U.plot(x_grid, U_pair, '--', color='tab:blue', label='Парное взаимодействие')
ax_U.plot(x_grid, U_trap, ':', color='tab:green', label='Гармоническое удержание')
ax_U.plot(x_grid, U_total, color='black', lw=2, label='Полный потенциал U')
ax_U.axvline(rc, color='red', ls='--', label='rc')
if visual_mode == 'switch':
    ax_U.axvspan(ron, rc, color='gold', alpha=0.15, label='Область переключения')
ax_U.axhline(e_vis[0], color='gray', ls=':', label='Начальная полная энергия')
energy_line = ax_U.axhline(e_vis[0], color='tab:orange', ls='--')
particle_U, = ax_U.plot([], [], 'o', color='tab:purple', ms=9)
kinetic_line, = ax_U.plot([], [], color='tab:orange', lw=4, label='K = E − U')
ax_U.set(xlim=(0.30, 1.05), ylim=(-1.2, 3.0),
         xlabel='x = r, nm', ylabel='Энергия, kJ/mol',
         title='Точка на U(x) отмечает текущую координату; отрезок показывает K')
ax_U.legend(loc='upper right', fontsize=8, ncol=2)

# Средняя панель: реальное расположение центров вдоль x.
ax_x.axhline(0, color='gray', lw=1)
ax_x.scatter([0], [0], s=160, marker='s', color='black', label='Неподвижный центр')
ax_x.axvline(0.65, color='tab:green', ls=':', label='Центр удержания')
ax_x.axvline(rc, color='red', ls='--', label='rc')
particle_x, = ax_x.plot([], [], 'o', color='tab:purple', ms=13)
force_arrow = ax_x.annotate('', xy=(0.6, 0.1), xytext=(0.55, 0.1),
                           arrowprops={'arrowstyle': '->', 'color': 'red', 'lw': 2})
ax_x.set(xlim=(-0.07, 1.08), ylim=(-0.16, 0.30), yticks=[], xlabel='x, nm',
         title='Частица движется по прямой; красная стрелка — полная сила')
ax_x.legend(loc='upper left', fontsize=8, ncol=3)
status_text = ax_x.text(0.02, 0.05, '', transform=ax_x.transAxes, fontsize=9)

# Нижняя панель: изменение полной механической энергии.
ax_E.plot(t_vis, e_vis-e_vis[0], color='gray', alpha=0.5)
energy_point, = ax_E.plot([], [], 'o', color='tab:orange')
ax_E.set(xlabel='t, ps', ylabel='E − E(0), kJ/mol', title='Ошибка полной энергии')
plt.tight_layout(rect=(0, 0, 1, 0.96))

def update_visual(j):
    particle_U.set_data([x_vis[j]], [u_vis[j]])
    kinetic_line.set_data([x_vis[j], x_vis[j]], [u_vis[j], e_vis[j]])
    energy_line.set_ydata([e_vis[j], e_vis[j]])
    particle_x.set_data([x_vis[j]], [0])
    # Длина стрелки сжата tanh для читаемости; точная сила дана в подписи.
    force_arrow.set_position((x_vis[j], 0.10))
    force_arrow.xy = (x_vis[j] + 0.13*np.tanh(f_vis[j]/5), 0.10)
    status_text.set_text(
        f't={t_vis[j]:.2f} ps | x={x_vis[j]:.3f} nm | '
        f'v={v_vis[j]:+.3f} nm/ps | F={f_vis[j]:+.2f} kJ/mol/nm'
    )
    energy_point.set_data([t_vis[j]], [e_vis[j]-e_vis[0]])
    return particle_U, kinetic_line, energy_line, particle_x, force_arrow, status_text, energy_point

frame_ids = np.unique(np.linspace(0, len(records)-1, 100).astype(int))
anim_vis = FuncAnimation(fig_vis, update_visual, frames=frame_ids,
                        interval=80, blit=False)
html_vis = anim_vis.to_jshtml()
plt.close(fig_vis)
display(HTML(html_vis))
print('Стрелка силы имеет условную длину. Верхняя панель — график энергии, а не движение по вертикали.')


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4)); checks={}
for kind in exprs:
    ctx,it=cutoff_context(kind,trap=True)
    ctx.setPositions([[0,0,0],[.55,0,0]]);ctx.setVelocities([[0,0,0],[.22,0,0]])
    vals=[]
    for j in range(12001):
        x,v,u=state(ctx);vals.append([j*.001,x[1,0],u+.5*40*np.sum(v[1]**2)])
        it.step(1)
    vals=np.array(vals); checks[kind]=np.ptp(vals[:,2])
    ax[0].plot(vals[:,0],vals[:,1],label=kind);ax[1].plot(vals[:,0],vals[:,2]-vals[0,2],label=kind)
    del ctx,it
ax[0].axhline(rc,color='gray',ls=':');ax[0].set(xlabel='t, ps',ylabel='r, nm')
ax[1].set(xlabel='t, ps',ylabel='E−E₀, kJ/mol');ax[1].legend();plt.show()
print('Размах E:',checks)
assert checks['switch']<checks['cut']


## Визуализация: какая система движется и почему она возвращается?

В точке $x=0$ закреплён неподвижный центр. В OpenMM нулевая масса исключает его из динамики. Вторая частица имеет массу 40 Da и стартует при $x=0.55$ nm со скоростью $+0.22$ nm/ps. Начальные $y,z$ и поперечные скорости равны нулю, поэтому движение остаётся вдоль оси $x$, а на этой траектории $r=x>0$.

На подвижную частицу действуют **два потенциала**:

$$
U_{\mathrm{total}}(x)=U_{\mathrm{pair}}(x)+\frac{k_{\mathrm{trap}}}{2}(x-x_{\mathrm{trap}})^2,
\qquad k_{\mathrm{trap}}=20\ \mathrm{kJ\,mol^{-1}\,nm^{-2}},\quad x_{\mathrm{trap}}=0.65\ \mathrm{nm}.
$$

Парная часть основана на потенциале Леннарда–Джонса с $\sigma=0.34$ nm и $\varepsilon=1$ kJ/mol. Параметр `visual_mode` выбирает уже определённые выше варианты `cut`, `shift` или `switch`, с $r_c=0.70$ nm и началом переключения $r_{\mathrm{on}}=0.50$ nm. **Отсечение применяется только к парной части, не к удерживающему потенциалу.**

$$
F_x=-\frac{dU_{\mathrm{pair}}}{dx}-k_{\mathrm{trap}}(x-x_{\mathrm{trap}}).
$$

За пределами $r_c$ парная сила равна нулю, но сила удержания продолжает действовать. Частица замедляется, достигает точки поворота и возвращается. При сближении с неподвижным центром быстро растущее LJ-отталкивание предотвращает слишком малые расстояния. Само удержание консервативно: это не трение и не термостат.

**Как читать анимацию:**

- Сверху показаны обе части потенциала и их сумма. Точка отмечает $U(x(t))$, оранжевый вертикальный отрезок — кинетическую энергию $K=E-U$. Высота точки на этом графике не является пространственной координатой частицы.
- Посередине показано реальное движение вдоль прямой: неподвижный центр, подвижная частица, центр удержания и граница отсечения. Красная стрелка показывает направление полной силы; её длина сжата для удобства, численное значение силы дано в подписи.
- Снизу показано изменение полной энергии. При `cut` пересечение разрыва потенциала даёт скачок энергии без соответствующего импульса скорости. При `shift` потенциал непрерывен, но сила на границе разрывна; `switch` плавно выключает парное взаимодействие.

Запустите сначала `cut`, затем измените `visual_mode` на `shift` и `switch`. Это новые расчёты с одинаковыми начальными координатами и скоростью; начальная потенциальная энергия и последующие траектории могут различаться. Нижняя панель нормирована на собственную начальную энергию каждого расчёта.


## Самостоятельная работа

1. Уменьшите шаг в два раза в NVE-части (и исправьте ось времени). Какие ошибки уменьшаются?
2. Сдвиньте ron ближе к rc: почему переключение становится резче?
3. Почему гладкое отсечение всё равно не решает проблему дальней электростатики?

**Подсказка:** гладкость — численное свойство, но потерянный дальний вклад остаётся потерянным.

## Источники и границы модели

[OpenMM: стандартные силы](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html); [Custom Forces](https://docs.openmm.org/latest/userguide/theory/03_custom_forces.html).